In [0]:
%sql
describe workspace.pj_sales.tb_fact_sale_gold

In [0]:
%sql
select
  tfs._sk_store
  ,tdst.store_id
  ,tdst.store_name
  ,tdst.store_group
  ,count(distinct tfs.invoice_id)                                                 as sales_quantity
  ,round(sum(tfs.part_total_price), 2)                                            as billing
  ,round(sum(tfs.part_profit), 2)                                                 as profit
  ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
from workspace.pj_sales.tb_fact_sale_gold tfs
inner join workspace.pj_sales.tb_dim_store_gold tdst
  on tdst._sk_store = tfs._sk_store
group by
  tfs._sk_store
  ,tdst.store_id
  ,tdst.store_name
  ,tdst.store_group

In [0]:
%sql
with
  aggregated_store as (
    select
      tfs._sk_store
      ,tdst.store_id
      ,tdst.store_name
      ,tdst.store_group
      ,count(distinct tfs.invoice_id)                                                 as sales_quantity
      ,round(sum(tfs.part_total_price), 2)                                            as billing
      ,round(sum(tfs.part_profit), 2)                                                 as profit
      ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
    from workspace.pj_sales.tb_fact_sale_gold tfs
    inner join workspace.pj_sales.tb_dim_store_gold tdst
      on tdst._sk_store = tfs._sk_store
    group by
      tfs._sk_store
      ,tdst.store_id
      ,tdst.store_name
      ,tdst.store_group
  )
select
  ast.store_id
  ,ast.store_name
  ,ast.store_group
  ,ast.sales_quantity
  ,ast.billing
  ,ast.profit
  ,ast.product_quantity
  ,dense_rank() over (order by ast.billing        desc)                               as rank_by_billing
  ,dense_rank() over (order by ast.profit         desc)                               as rank_by_profit
  ,dense_rank() over (order by ast.sales_quantity desc)                               as rank_by_sales_quantity
from aggregated_store ast
order by 
   ast._sk_store
  ,ast.billing                                                                desc

In [0]:
%sql
create or replace view workspace.pj_sales.vw_rank_by_store as (
  with
    aggregated_store as (
      select
        tfs._sk_store
        ,tdst.store_id
        ,tdst.store_name
        ,tdst.store_group
        ,count(distinct tfs.invoice_id)                                                 as sales_quantity
        ,round(sum(tfs.part_total_price), 2)                                            as billing
        ,round(sum(tfs.part_profit), 2)                                                 as profit
        ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
      from workspace.pj_sales.tb_fact_sale_gold tfs
      inner join workspace.pj_sales.tb_dim_store_gold tdst
        on tdst._sk_store = tfs._sk_store
      group by
        tfs._sk_store
        ,tdst.store_id
        ,tdst.store_name
        ,tdst.store_group
    )
  select
    ast.store_id
    ,ast.store_name
    ,ast.store_group
    ,ast.sales_quantity
    ,ast.billing
    ,ast.profit
    ,ast.product_quantity
    ,dense_rank() over (order by ast.billing        desc)                               as rank_by_billing
    ,dense_rank() over (order by ast.profit         desc)                               as rank_by_profit
    ,dense_rank() over (order by ast.sales_quantity desc)                               as rank_by_sales_quantity
  from aggregated_store ast
)